# Retrieval-budget proxy experiment

This notebook compares Vector RAG with 50 retrieved chunks (`b50`) and 200 retrieved chunks (`b200`) as a **known-direction sanity check** for calibrated absolute scoring. It also measures agreement with pairwise and assertion scoring.

> `b200` is weak ground truth, not human ground truth. More context can add noise, so the expected direction is `b200 >= b50`, especially when the additional context contributes useful evidence.

In [ ]:
# Copyright (c) 2025 Microsoft Corporation.
from __future__ import annotations

import asyncio
import functools
import json
import operator
import os
from pathlib import Path
from typing import Any, cast

import nest_asyncio
import numpy as np
import pandas as pd
from graphrag_common.config import load_config
from IPython.display import display as show
from litellm.exceptions import APIConnectionError
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score

import benchmark_qed
from benchmark_qed.autoe.assertion import get_assertion_scores
from benchmark_qed.autoe.calibrated import answer_items_from_frame, score_answers
from benchmark_qed.autoe.config import CalibratedAbsoluteScoringConfig
from benchmark_qed.autoe.pairwise import get_pairwise_scores
from benchmark_qed.cache import create_default_cache_config
from benchmark_qed.config.model.score import Criteria
from benchmark_qed.llm.factory import ModelFactory

nest_asyncio.apply()

## 1. Configure the experiment

The defaults use the repository's paired `data_global` examples because global questions provide the stronger retrieval-budget sensitivity test. Set `SCORING_PROJECT` to the absolute-scoring project containing `settings.yaml` and the frozen calibration path. Change `QUESTION_SET` to `data_local` to rerun the lower-effect local control; outputs and caches are isolated by question set.

In [ ]:
SCORING_PROJECT = Path(
    "<path to absolute scoring>/absolute_scoring"
)
SETTINGS_PATH = SCORING_PROJECT / "settings.yaml"

REPOSITORY_ROOT = Path(benchmark_qed.__file__).resolve().parents[1]
EXAMPLE_DIR = REPOSITORY_ROOT / "docs/notebooks/example_answers"
QUESTION_SET = "data_global"
# Neutral condition names prevent the judge from seeing the retrieval budgets.
B50_NAME = "condition_a"
B200_NAME = "condition_b"
B50_PATH = EXAMPLE_DIR / "vector_rag_short_context" / f"{QUESTION_SET}_answers.json"
B200_PATH = EXAMPLE_DIR / "vector_rag_long_context" / f"{QUESTION_SET}_answers.json"
ASSERTIONS_PATH = EXAMPLE_DIR / f"{QUESTION_SET}_assertions.json"

OUTPUT_DIR = SCORING_PROJECT / "output/budget_proxy_experiment" / QUESTION_SET
ABSOLUTE_PATH = OUTPUT_DIR / "absolute_scores.csv"
PAIRWISE_PATH = OUTPUT_DIR / "pairwise_scores.csv"
ASSERTION_PATH = OUTPUT_DIR / "assertion_scores.csv"
DIRECTION_PATH = OUTPUT_DIR / "directional_comparison.csv"
SUMMARY_PATH = OUTPUT_DIR / "directional_summary.csv"

ABSOLUTE_PASSES = 3
PAIRWISE_TRIALS = 4
ASSERTION_TRIALS = 2
MAX_QUESTIONS: int | None = None

RUN_ABSOLUTE = True
RUN_PAIRWISE = True
RUN_ASSERTION = True

## 2. Protocol and validity checks

The notebook verifies paired IDs, exact 50/200 retrieval budgets, and no question-ID overlap with calibration. Evaluators see neutral condition names rather than the budget values. All methods use the same configured judge model.

Outputs are converted to a common direction: `-1 = b50`, `0 = tie`, `+1 = b200`. Assertion scoring is an overall content-coverage signal; it is not criterion-equivalent to the four absolute/pairwise criteria.

In [ ]:
working_directory = Path.cwd()
try:
    config = load_config(CalibratedAbsoluteScoringConfig, SCORING_PROJECT)
finally:
    os.chdir(working_directory)
if config.input_storage is not None or config.output_storage is not None:
    msg = "This experiment notebook currently supports local storage only."
    raise ValueError(msg)

calibration_path = Path(config.calibration_path)
if not calibration_path.is_absolute():
    calibration_path = SCORING_PROJECT / calibration_path
calibration: dict[str, Any] = json.loads(calibration_path.read_text(encoding="utf-8"))
criteria = [
    Criteria.model_validate(scale["criterion"])
    for scale in calibration["scales"].values()
]

b50 = pd.read_json(B50_PATH)
b200 = pd.read_json(B200_PATH)
if MAX_QUESTIONS is not None:
    selected_ids = set(b50["question_id"].astype(str).head(MAX_QUESTIONS))
    b50 = b50.loc[b50["question_id"].astype(str).isin(selected_ids)].copy()
    b200 = b200.loc[b200["question_id"].astype(str).isin(selected_ids)].copy()

b50_ids = set(b50["question_id"].astype(str))
b200_ids = set(b200["question_id"].astype(str))
if b50_ids != b200_ids:
    msg = "b50 and b200 must contain exactly the same question IDs."
    raise ValueError(msg)
if b50["question_id"].duplicated().any() or b200["question_id"].duplicated().any():
    msg = "Each budget condition must contain one answer per question ID."
    raise ValueError(msg)

retrieval_sizes: dict[str, set[int]] = {}
for name, directory in [
    (B50_NAME, "vector_rag_short_context"),
    (B200_NAME, "vector_rag_long_context"),
]:
    records = json.loads(
        (EXAMPLE_DIR / directory / f"{QUESTION_SET}_retrieval_results.json").read_text(
            encoding="utf-8"
        )
    )
    retrieval_sizes[name] = {len(record["context"]) for record in records}
if retrieval_sizes[B50_NAME] != {50} or retrieval_sizes[B200_NAME] != {200}:
    msg = f"Unexpected retrieval budgets: {retrieval_sizes}"
    raise ValueError(msg)

calibration_ids = {str(item["question_id"]) for item in calibration["items"].values()}
overlap = sorted(calibration_ids & b50_ids)
if overlap:
    msg = f"Evaluation leaks {len(overlap)} calibration question IDs."
    raise ValueError(msg)

paired_inputs = cast(
    pd.DataFrame,
    b50[["question_id", "answer"]].merge(
        b200[["question_id", "answer"]],
        on="question_id",
        suffixes=("_b50", "_b200"),
        validate="one_to_one",
    ),
)
paired_inputs["b50_words"] = paired_inputs["answer_b50"].str.split().str.len()
paired_inputs["b200_words"] = paired_inputs["answer_b200"].str.split().str.len()
paired_inputs["word_gain"] = paired_inputs["b200_words"] - paired_inputs["b50_words"]

print(f"Model: {config.llm_config.model}")
print(f"Provider: {config.llm_config.llm_provider}")
print(f"Paired questions: {len(b50_ids)}")
print(f"Criteria: {', '.join(criterion.name for criterion in criteria)}")
print(f"Calibration overlap: {len(overlap)}")
show(paired_inputs[["b50_words", "b200_words", "word_gain"]].describe())

## 3. Run evaluators

> These cells make paid model calls only when their corresponding flag is `True`. With 50 questions and the default settings, absolute scoring requests 1,200 classifications, pairwise requests 800 judgments, and assertion scoring requests two trials for every assertion under both conditions. Run one evaluator at a time and keep the caches.

In [ ]:
if RUN_ABSOLUTE:
    llm_client = ModelFactory.create_chat_model(config.llm_config)
    absolute_targets = [
        *answer_items_from_frame(
            b50.rename(columns={"question": "question_text"}),
            condition=B50_NAME,
        ),
        *answer_items_from_frame(
            b200.rename(columns={"question": "question_text"}),
            condition=B200_NAME,
        ),
    ]
    total_per_criterion = len(absolute_targets) * ABSOLUTE_PASSES
    total = total_per_criterion * len(criteria)
    interval = max(1, total_per_criterion // 20)
    completed_by_criterion = dict.fromkeys(
        (criterion.name for criterion in criteria), 0
    )
    print(
        f"Starting absolute scoring: {total} classifications "
        f"({total_per_criterion} per criterion).",
        flush=True,
    )

    def absolute_complete(criterion_name: str) -> None:
        """Print notebook-safe absolute-scoring progress."""
        completed = completed_by_criterion[criterion_name] + 1
        completed_by_criterion[criterion_name] = completed
        if completed % interval == 0 or completed == total_per_criterion:
            print(
                f"Absolute {criterion_name}: {completed}/{total_per_criterion} "
                f"({100 * completed / total_per_criterion:.0f}%)",
                flush=True,
            )

    try:
        absolute_records = asyncio.get_event_loop().run_until_complete(
            score_answers(
                llm=llm_client,
                llm_config=config.llm_config,
                targets=absolute_targets,
                calibration=calibration,
                passes=ABSOLUTE_PASSES,
                cache_config=config.cache_config,
                complete_callback=absolute_complete,
            )
        )
    except APIConnectionError:
        print(
            "Azure disconnected. Valid completed judgments are cached; rerun this "
            "cell to resume. If disconnects repeat, lower concurrent_requests in "
            "settings.yaml.",
            flush=True,
        )
        raise
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(absolute_records).to_csv(ABSOLUTE_PATH, index=False)
    print(f"Saved absolute scores to {ABSOLUTE_PATH}", flush=True)
else:
    print("Absolute scoring skipped. Set RUN_ABSOLUTE = True to run it.")

In [ ]:
if RUN_PAIRWISE:
    llm_client = ModelFactory.create_chat_model(config.llm_config)
    pairwise_total_per_criterion = len(b50_ids) * PAIRWISE_TRIALS
    pairwise_total = pairwise_total_per_criterion * len(criteria)
    pairwise_interval = max(1, pairwise_total_per_criterion // 20)
    pairwise_completed = dict.fromkeys((criterion.name for criterion in criteria), 0)
    print(
        f"Starting pairwise scoring: {pairwise_total} judgments "
        f"({pairwise_total_per_criterion} per criterion).",
        flush=True,
    )

    def pairwise_complete(criterion_name: str) -> None:
        """Print notebook-safe pairwise progress."""
        completed = pairwise_completed[criterion_name] + 1
        pairwise_completed[criterion_name] = completed
        if (
            completed % pairwise_interval == 0
            or completed == pairwise_total_per_criterion
        ):
            print(
                f"Pairwise {criterion_name}: {completed}/{pairwise_total_per_criterion} "
                f"({100 * completed / pairwise_total_per_criterion:.0f}%)",
                flush=True,
            )

    try:
        pairwise_scores = get_pairwise_scores(
            llm_client=llm_client,
            llm_config=config.llm_config,
            base_name="Answer A",
            other_name="Answer B",
            base_answers=b50,
            other_answers=b200,
            criteria=criteria,
            trials=PAIRWISE_TRIALS,
            include_score_id_in_prompt=True,
            question_text_key="question",
            cache_config=create_default_cache_config(
                OUTPUT_DIR / ".cache/pairwise", database_name="pairwise.sqlite3"
            ),
            show_progress=False,
            complete_callback=pairwise_complete,
        )
    except APIConnectionError:
        print(
            "Azure disconnected. Valid completed judgments are cached; rerun this "
            "cell to resume. If disconnects repeat, lower concurrent_requests in "
            "settings.yaml.",
            flush=True,
        )
        raise
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pairwise_scores.to_csv(PAIRWISE_PATH, index=False)
    print(f"Saved pairwise scores to {PAIRWISE_PATH}", flush=True)
else:
    print("Pairwise scoring skipped. Set RUN_PAIRWISE = True to run it.")

In [ ]:
assertion_source = pd.read_json(ASSERTIONS_PATH)
assertion_columns = cast(
    pd.DataFrame,
    assertion_source[["question_id", "question_text", "assertions"]],
)
assertions = cast(pd.DataFrame, assertion_columns.explode("assertions"))
assertions["assertion"] = assertions["assertions"].map(operator.itemgetter("statement"))
assertions = assertions.loc[
    assertions["question_id"].astype(str).isin(list(b50_ids)),
    ["question_id", "question_text", "assertion"],
]

if RUN_ASSERTION:
    llm_client = ModelFactory.create_chat_model(config.llm_config)
    assertion_total_per_condition = len(assertions) * ASSERTION_TRIALS
    expected = assertion_total_per_condition * 2
    assertion_interval = max(1, assertion_total_per_condition // 20)
    assertion_completed: dict[str, int] = dict.fromkeys((B50_NAME, B200_NAME), 0)
    print(
        f"Starting assertion scoring: {expected} judgments "
        f"({assertion_total_per_condition} per condition).",
        flush=True,
    )

    def assertion_complete(condition_name: str) -> None:
        """Print notebook-safe assertion progress."""
        completed = assertion_completed[condition_name] + 1
        assertion_completed[condition_name] = completed
        if (
            completed % assertion_interval == 0
            or completed == assertion_total_per_condition
        ):
            print(
                f"Assertion {condition_name}: {completed}/{assertion_total_per_condition} "
                f"({100 * completed / assertion_total_per_condition:.0f}%)",
                flush=True,
            )

    assertion_frames = []
    for condition_name, answers in [(B50_NAME, b50), (B200_NAME, b200)]:
        try:
            frame = get_assertion_scores(
                llm_client=llm_client,
                llm_config=config.llm_config,
                answers=answers,
                assertions=assertions,
                trials=ASSERTION_TRIALS,
                question_text_key="question",
                cache_config=create_default_cache_config(
                    OUTPUT_DIR / ".cache/assertion",
                    database_name="assertion.sqlite3",
                ),
                show_progress=False,
                complete_callback=functools.partial(
                    assertion_complete, condition_name
                ),
            )
        except APIConnectionError:
            print(
                "Azure disconnected. Valid completed judgments are cached; rerun "
                "this cell to resume. If disconnects repeat, lower "
                "concurrent_requests in settings.yaml.",
                flush=True,
            )
            raise
        frame["condition"] = condition_name
        assertion_frames.append(frame)
    assertion_scores = pd.concat(assertion_frames, ignore_index=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    assertion_scores.to_csv(ASSERTION_PATH, index=False)
    print(f"Saved assertion scores to {ASSERTION_PATH}", flush=True)
else:
    print("Assertion scoring skipped. Set RUN_ASSERTION = True to run it.")

## 4. Normalize outputs to `b50 / tie / b200`

Run this section after all three CSVs exist. Absolute and pairwise preferences are retained per criterion. Three-way agreement uses an overall absolute margin and overall pairwise win margin because assertion coverage has no matching criterion labels.

In [ ]:
required_outputs = [ABSOLUTE_PATH, PAIRWISE_PATH, ASSERTION_PATH]
missing_outputs = [str(path) for path in required_outputs if not path.exists()]
if missing_outputs:
    msg = "Run the missing evaluators first: " + ", ".join(missing_outputs)
    raise FileNotFoundError(msg)

absolute_raw = pd.read_csv(ABSOLUTE_PATH)
pairwise_raw = pd.read_csv(PAIRWISE_PATH)
assertion_raw = pd.read_csv(ASSERTION_PATH)

absolute_levels = cast(
    pd.DataFrame,
    absolute_raw.pivot_table(
        index=["question_id", "criteria"],
        columns="condition",
        values="level",
        aggfunc="first",
    ).reset_index(),
)
absolute_levels["absolute_margin"] = (
    absolute_levels[B200_NAME] - absolute_levels[B50_NAME]
)
absolute_levels["absolute_preference"] = np.sign(
    absolute_levels["absolute_margin"]
).astype("Int64")

question_ids = b50[["question_id", "question"]].rename(
    columns={"question": "question_text"}
)  # type: ignore[call-overload] - pandas stubs reject a valid columns mapping
pairwise_with_ids = cast(
    pd.DataFrame,
    pairwise_raw.merge(
        question_ids,
        left_on="question",
        right_on="question_text",
        validate="many_to_one",
    ),
)
pairwise_items = cast(
    pd.DataFrame,
    pairwise_with_ids.groupby(["question_id", "criteria"], as_index=False).agg(
        b50_pairwise=("Answer A_score", "mean"),
        b200_pairwise=("Answer B_score", "mean"),
    ),
)
pairwise_items["pairwise_margin"] = (
    pairwise_items["b200_pairwise"] - pairwise_items["b50_pairwise"]
)
pairwise_items["pairwise_preference"] = np.sign(
    pairwise_items["pairwise_margin"]
).astype("Int64")

assertion_with_ids = cast(
    pd.DataFrame,
    assertion_raw.merge(
        question_ids,
        left_on="question",
        right_on="question_text",
        validate="many_to_one",
    ),
)
assertion_means = cast(
    pd.DataFrame,
    assertion_with_ids.groupby(["question_id", "condition"], as_index=False)[
        "score"
    ].mean(),
)
assertion_items = cast(
    pd.DataFrame,
    assertion_means.pivot_table(
        index="question_id",
        columns="condition",
        values="score",
        aggfunc="first",
    ).reset_index(),
)
assertion_items["assertion_margin"] = (
    assertion_items[B200_NAME] - assertion_items[B50_NAME]
)
assertion_items["assertion_preference"] = np.sign(
    assertion_items["assertion_margin"]
).astype("Int64")

criterion_comparison = cast(
    pd.DataFrame,
    absolute_levels.merge(
        pairwise_items,
        on=["question_id", "criteria"],
        validate="one_to_one",
    ),
)
overall = cast(
    pd.DataFrame,
    criterion_comparison.groupby("question_id", as_index=False).agg(
        absolute_margin=("absolute_margin", "mean"),
        pairwise_margin=("pairwise_margin", "mean"),
    ),
).merge(
    assertion_items[["question_id", "assertion_margin", "assertion_preference"]],
    on="question_id",
    validate="one_to_one",
)
overall["absolute_preference"] = np.sign(overall["absolute_margin"]).astype("Int64")
overall["pairwise_preference"] = np.sign(overall["pairwise_margin"]).astype("Int64")
overall = overall.merge(
    paired_inputs[["question_id", "word_gain"]],
    on="question_id",
    validate="one_to_one",
)
overall.to_csv(DIRECTION_PATH, index=False)
show(overall.head())

In [ ]:
preference_columns = {
    "absolute": "absolute_preference",
    "pairwise": "pairwise_preference",
    "assertion": "assertion_preference",
}
summary_records = []
for method, column in preference_columns.items():
    values = overall[column].dropna().astype(int)
    summary_records.append({
        "method": method,
        "questions": len(values),
        "b200_win_rate": float((values == 1).mean()),
        "tie_rate": float((values == 0).mean()),
        "b50_inversion_rate": float((values == -1).mean()),
    })
proxy_summary = pd.DataFrame(summary_records)
proxy_summary.to_csv(SUMMARY_PATH, index=False)
show(proxy_summary)

In [ ]:
agreement_records = []
methods = list(preference_columns)
for index, left in enumerate(methods):
    for right in methods[index + 1 :]:
        columns = [preference_columns[left], preference_columns[right]]
        comparable = overall[columns].dropna().astype(int)
        left_values = comparable[columns[0]]
        right_values = comparable[columns[1]]
        agreement_records.append({
            "methods": f"{left} vs {right}",
            "questions": len(comparable),
            "excluded_missing": len(overall) - len(comparable),
            "exact_direction_agreement": float((left_values == right_values).mean()),
            "cohen_kappa": float(cohen_kappa_score(left_values, right_values)),
        })
agreement = pd.DataFrame(agreement_records)
preference_column_names = list(preference_columns.values())
complete_preferences = overall[preference_column_names].dropna().astype(int)
preference_array = complete_preferences.to_numpy()
overall["three_way_comparable"] = overall.index.isin(complete_preferences.index)
overall["three_way_unanimous"] = False
overall["two_or_more_agree"] = False
overall.loc[complete_preferences.index, "three_way_unanimous"] = (
    (preference_array[:, 0] == preference_array[:, 1])
    & (preference_array[:, 1] == preference_array[:, 2])
)
overall.loc[complete_preferences.index, "two_or_more_agree"] = (
    (preference_array[:, 0] == preference_array[:, 1])
    | (preference_array[:, 0] == preference_array[:, 2])
    | (preference_array[:, 1] == preference_array[:, 2])
)
excluded = len(overall) - len(complete_preferences)
print(
    f"Three-way comparable questions: {len(complete_preferences)}/{len(overall)} "
    f"({excluded} excluded due to missing preferences)"
)
comparable_overall = overall.loc[overall["three_way_comparable"]]
print(f"Three-way unanimous: {comparable_overall['three_way_unanimous'].mean():.1%}")
print(f"At least two agree: {comparable_overall['two_or_more_agree'].mean():.1%}")
show(agreement)

def summarize_criterion_agreement(frame: pd.DataFrame) -> pd.Series:
    comparable = frame.dropna(
        subset=["absolute_preference", "pairwise_preference"]
    )
    return pd.Series({
        "questions": len(comparable),
        "excluded_missing": len(frame) - len(comparable),
        "absolute_pairwise_agreement": (
            comparable["absolute_preference"]
            == comparable["pairwise_preference"]
        ).mean(),
        "absolute_b200_win_rate": (
            comparable["absolute_preference"] == 1
        ).mean(),
        "pairwise_b200_win_rate": (
            comparable["pairwise_preference"] == 1
        ).mean(),
    })


criterion_agreement = cast(
    pd.DataFrame,
    criterion_comparison
    .groupby("criteria")
    .apply(summarize_criterion_agreement, include_groups=False)
    .reset_index(),
)
show(criterion_agreement)

In [ ]:
margin_columns = ["absolute_margin", "pairwise_margin", "assertion_margin"]

def spearman_record(
    label: str, left_values: pd.Series, right_values: pd.Series
) -> dict[str, Any]:
    paired = pd.concat([left_values, right_values], axis=1).dropna()
    if len(paired) < 2:
        return {
            "margins": label,
            "observations": len(paired),
            "spearman": float("nan"),
            "p_value": float("nan"),
            "status": "undefined: fewer than 2 observations",
        }
    if paired.iloc[:, 0].nunique() < 2 or paired.iloc[:, 1].nunique() < 2:
        return {
            "margins": label,
            "observations": len(paired),
            "spearman": float("nan"),
            "p_value": float("nan"),
            "status": "undefined: constant input",
        }
    result = cast(Any, spearmanr(paired.iloc[:, 0], paired.iloc[:, 1]))
    return {
        "margins": label,
        "observations": len(paired),
        "spearman": float(result.statistic),
        "p_value": float(result.pvalue),
        "status": "ok",
    }


correlations = []
for index, left in enumerate(margin_columns):
    for right in margin_columns[index + 1 :]:
        correlations.append(
            spearman_record(
                f"{left} vs {right}", overall[left], overall[right]
            )
        )
for margin in margin_columns:
    correlations.append(
        spearman_record(
            f"word_gain vs {margin}", overall["word_gain"], overall[margin]
        )
    )
show(pd.DataFrame(correlations))

print("Cases where every evaluator unexpectedly prefers b50")
unanimous_b50 = overall.loc[
    (overall["absolute_preference"] == -1)
    & (overall["pairwise_preference"] == -1)
    & (overall["assertion_preference"] == -1)
]
show(unanimous_b50.sort_values("word_gain", ascending=False))

print("Three-method disagreements for manual adjudication")
show(
    overall.loc[
        overall["three_way_comparable"] & ~overall["three_way_unanimous"]
    ]
)

## 5. Interpretation

- A high `b200_win_rate` is evidence that an evaluator detects the expected retrieval-budget intervention; it is not answer-level accuracy.
- `b50_inversion_rate` is the most important sanity-check failure rate, but individual inversions may be legitimate when additional context adds noise.
- Absolute/pairwise criterion agreement is directly interpretable because both use the same criteria. Assertion agreement is only overall content-coverage agreement.
- Strong correlation with `word_gain` may indicate verbosity bias rather than quality sensitivity.
- Inspect unanimous `b50` cases and three-method disagreements manually. The next stronger study should label whether the extra 150 chunks add relevant evidence and test whether evaluator margins increase with that evidence gain.
- Because all methods may use the same LLM judge, agreement can reflect correlated judge bias. A blinded human subset or a second judge model is needed for stronger validation.